In [ ]:
# # For kaggle:
# !git clone https://github.com/maTiahK/CORT_SI.git
# %cd /kaggle/working/CORT_SI

# !python -m pip install -q --upgrade pip
# !python -m pip install -q -e .


# from cort_si import SI, SI_randj, gen_data
# print("CORT_SI import OK")

Cloning into 'CORT_SI'...
remote: Enumerating objects: 74, done.
remote: Counting objects: 100% (74/74), done.
remote: Compressing objects: 100% (57/57), done.
remote: Total 74 (delta 19), reused 69 (delta 14), pack-reused 0 (from 0)
Receiving objects: 100% (74/74), 258.65 KiB | 4.54 MiB/s, done.
Resolving deltas: 100% (19/19), done.
/kaggle/working/CORT_SI
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 21.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for cort-si (pyproject.toml) ... done
CORT_SI import OK


# Example 1: Computing p-values with Adaptive CoRT-SI

This notebook demonstrates how to use the `cort_si` package to compute selective inference p-values for Adaptive CoRT-SI.

In [2]:
import random
import sys

import numpy as np

sys.path.append('..')

from cort_si import SI, SI_randj, gen_data

## 1. Generate Synthetic Data

In [ ]:
np.random.seed(0)
random.seed(0)

p = 300
nS = 100
nT = 50
K = 2
true_beta_scale = 1.0
rho = 0.0
sigma_noise = 1.0
source_shift_sd = 0.3
covariate_shift = False
seed = 0

XS_list, YS_list, X0, Y0, _, SigmaS_list, Sigma0, beta0 = gen_data.generate_data(
    p=p,
    nS=nS,
    nT=nT,
    K=K,
    true_beta=true_beta_scale,
    rho=rho,
    sigma_noise=sigma_noise,
    source_shift_sd=source_shift_sd,
    covariate_shift=covariate_shift,
    seed=seed,
)

print(f'Number of auxiliary groups: {len(XS_list)}')
print(f'Feature dimension: {p}')
print(f'Nonzero target coefficients: {int(np.count_nonzero(beta0))}')
print(f'Source sample size: {nS}')
print(f'Target sample size: {X0.shape[0]}')

Number of auxiliary groups: 2
Feature dimension: 300
Sparsity level: 5
Source sample size: 100
Target sample size: 50


## 2. Set Parameters

In [4]:
lambda_sel = 0.5
lambda0 = 0.5
lambdak_list = [0.5] * len(XS_list)
T = 3
verbose = True

print(f'lambda_sel: {lambda_sel}')
print(f'lambda0: {lambda0}')
print(f'T: {T}')
print('Truncation defaults: z_min=-20, z_max=20')
print(f'verbose: {verbose}')

lambda_sel: 0.5
lambda0: 0.5
T: 3
Truncation defaults: z_min=-20, z_max=20
verbose: True


## 3. Compute p-values for All Selected Features

In [5]:
p_values = SI(
    X0,
    Y0,
    XS_list,
    YS_list,
    lambda_sel=lambda_sel,
    lambda0=lambda0,
    lambdak_list=lambdak_list,
    SigmaS_list=SigmaS_list,
    Sigma0=Sigma0,
    T=T,
    verbose=verbose,
)

if p_values is not None:
    print(f'Number of selected features: {len(p_values)}')
    print('\nFeature index and p-values:')
    for j, p_val in p_values:
        print(f'Feature {j}: beta0[{j}] = {beta0[j]:.4f}, p-value = {p_val:.4f}')
else:
    print('No features selected')

source 0, fold 1 target-only: active set = [0, 2, 3, 4, 112, 153, 160, 198, 235]
source 0, fold 1 target+source: active set = [1, 2, 3, 4]
source 0, fold 1: vote = True
source 0, fold 2 target-only: active set = [2, 3, 4, 31, 125, 127, 153, 210, 269]
source 0, fold 2 target+source: active set = [1, 2, 3, 4]
source 0, fold 2: vote = True
source 0, fold 3 target-only: active set = [2, 4, 65, 100, 149, 191, 205, 210, 261]
source 0, fold 3 target+source: active set = [1, 2, 3, 4]
source 0, fold 3: vote = True
source 0: votes = 3/3
source 1, fold 1 target-only: active set = [0, 2, 3, 4, 112, 153, 160, 198, 235]
source 1, fold 1 target+source: active set = [0, 1, 2, 3, 4, 82]
source 1, fold 1: vote = True
source 1, fold 2 target-only: active set = [2, 3, 4, 31, 125, 127, 153, 210, 269]
source 1, fold 2 target+source: active set = [0, 1, 2, 3, 4]
source 1, fold 2: vote = True
source 1, fold 3 target-only: active set = [2, 4, 65, 100, 149, 191, 205, 210, 261]
source 1, fold 3 target+source: ac

## 4. Compute a p-value for a Random Selected Feature

This section reuses the same synthetic-data setup and parameter choice, then applies `SI_randj(...)` to one randomly chosen selected target feature.

In [6]:
rng_seed = 11

if p_values is None:
    print('No features selected')
else:
    selected_features = [j for j, _ in p_values]
    j_rand = random.Random(rng_seed).choice(selected_features)
    random.seed(rng_seed)
    p_value_rand = SI_randj(
        X0,
        Y0,
        XS_list,
        YS_list,
        lambda_sel=lambda_sel,
        lambda0=lambda0,
        lambdak_list=lambdak_list,
        SigmaS_list=SigmaS_list,
        Sigma0=Sigma0,
        T=T,
        verbose=verbose,
    )

    print(f'Number of selected features: {len(selected_features)}')
    print(f'Selected feature set: {selected_features}')
    if p_value_rand is not None:
        print(f'Random selected feature: {j_rand}')
        print(f'true beta0[{j_rand}] = {beta0[j_rand]:.4f}')
        print(f'p-value = {p_value_rand:.4f}')
    else:
        print('No valid truncation interval for the random selected feature')

source 0, fold 1 target-only: active set = [0, 2, 3, 4, 112, 153, 160, 198, 235]
source 0, fold 1 target+source: active set = [1, 2, 3, 4]
source 0, fold 1: vote = True
source 0, fold 2 target-only: active set = [2, 3, 4, 31, 125, 127, 153, 210, 269]
source 0, fold 2 target+source: active set = [1, 2, 3, 4]
source 0, fold 2: vote = True
source 0, fold 3 target-only: active set = [2, 4, 65, 100, 149, 191, 205, 210, 261]
source 0, fold 3 target+source: active set = [1, 2, 3, 4]
source 0, fold 3: vote = True
source 0: votes = 3/3
source 1, fold 1 target-only: active set = [0, 2, 3, 4, 112, 153, 160, 198, 235]
source 1, fold 1 target+source: active set = [0, 1, 2, 3, 4, 82]
source 1, fold 1: vote = True
source 1, fold 2 target-only: active set = [2, 3, 4, 31, 125, 127, 153, 210, 269]
source 1, fold 2 target+source: active set = [0, 1, 2, 3, 4]
source 1, fold 2: vote = True
source 1, fold 3 target-only: active set = [2, 4, 65, 100, 149, 191, 205, 210, 261]
source 1, fold 3 target+source: ac